Retrieving the data from final dataset.zip uploaded to drive and unzipping it to rain the yolo11n model

In [ ]:
import zipfile
import os

zip_file_path = '/content/drive/MyDrive/final_dataset-20260622T165852Z-3-001.zip'
output_dir = '/content/unzipped_data'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

print(f"File unzipped to: {output_dir}")
print(f"Contents of {output_dir}:\n{os.listdir(output_dir)}")

File unzipped to: /content/unzipped_data
Contents of /content/unzipped_data:
['final_dataset']


In [ ]:
!pip install ultralytics

GPU VERIFICATION FOR TRAINING THE MODEL


In [ ]:
!nvidia-smi

Mon Jun 22 17:37:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.12.1+cu130
True
Tesla T4


TRAIN TEST VALID SPLIT


In [ ]:
import yaml
import os

data_yaml_path = '/content/unzipped_data/final_dataset/data.yaml'

# Check if the data.yaml file exists
if not os.path.exists(data_yaml_path):
    print(f"Error: data.yaml not found at {data_yaml_path}")
else:
    # Read the current data.yaml content
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    # Update the 'path' in data.yaml to the correct unzipped directory
    data_config['path'] = '/content/unzipped_data/final_dataset'

    # Update train, val, test paths to be relative to the 'path' key
    # Assuming your dataset structure is like final_dataset/train/images, final_dataset/valid/images, etc.
    data_config['train'] = 'train/images'
    data_config['val'] = 'valid/images'
    data_config['test'] = 'test/images'

    # Save the updated data.yaml
    with open(data_yaml_path, 'w') as f:
        yaml.safe_dump(data_config, f)

    print(f"Updated {data_yaml_path} with path: {data_config['path']}")
    print("Updated data.yaml content:")
    print(yaml.dump(data_config, default_flow_style=False))

Updated /content/unzipped_data/final_dataset/data.yaml with path: /content/unzipped_data/final_dataset
Updated data.yaml content:
names:
  0: bottle
  1: cup
  2: remote
nc: 3
path: /content/unzipped_data/final_dataset
test: test/images
train: train/images
val: valid/images



In [ ]:
MODEL TRAINING (YOLO11N)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/unzipped_data/final_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project="/content/runs/detect",
    name="final_train"
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.12.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/unzipped_data/final_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=final_train-7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

SAVING THE TRAINED MODEL WITH BEST WEIGHTS

In [ ]:
!cp /content/runs/detect/final_train-7/weights/best.pt \
    /content/drive/MyDrive/

In [ ]:
!cp /content/runs/detect/final_train-7/weights/last.pt \
    /content/drive/MyDrive/

CHECKING WITH SAMPLE IMAGES BEFOR THE WEBCAM TESTING FOR DEBUGGING

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/final_train-7/weights/best.pt")

results = model.predict(
    source="/content/WIN_20260622_18_37_53_Pro.jpg",
    conf=0.01,
    save=True
)

for r in results:
    for box in r.boxes:
        cls = int(box.cls)
        conf = float(box.conf)

        print(model.names[cls], conf)


image 1/1 /content/WIN_20260622_18_37_53_Pro.jpg: 384x640 2 cups, 55.6ms
Speed: 2.2ms preprocess, 55.6ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/runs/detect/predict
cup 0.9074689745903015
cup 0.012550137005746365


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/final_train-7/weights/best.pt")

results = model.predict(
    source="/content/WIN_20260622_18_37_45_Pro.jpg",
    conf=0.01,
    save=True
)

for r in results:
    for box in r.boxes:
        cls = int(box.cls)
        conf = float(box.conf)

        print(model.names[cls], conf)


image 1/1 /content/WIN_20260622_18_37_45_Pro.jpg: 384x640 1 bottle, 4 cups, 11.7ms
Speed: 2.0ms preprocess, 11.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/runs/detect/predict-2
cup 0.6099156141281128
cup 0.19600403308868408
bottle 0.16336891055107117
cup 0.1377040445804596
cup 0.015518656000494957


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/final_train-7/weights/best.pt")

results = model.predict(
    source="/content/WIN_20260622_18_37_39_Pro.jpg",
    conf=0.01,
    save=True
)

for r in results:
    for box in r.boxes:
        cls = int(box.cls)
        conf = float(box.conf)

        print(model.names[cls], conf)


image 1/1 /content/WIN_20260622_18_37_39_Pro.jpg: 384x640 1 remote, 10.5ms
Speed: 2.1ms preprocess, 10.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/runs/detect/predict-3
remote 0.9037935733795166
